<a href="https://colab.research.google.com/github/Umama123/Machine-Learning-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Output directory setup
os.makedirs("work/outputs", exist_ok=True)

# 2. Connection setup & httpfs extension
conn = duckdb.connect()
conn.sql("INSTALL httpfs; LOAD httpfs;")

# 3. Colab Secrets se HF_TOKEN fetch karke DuckDB authentication set karein
try:
    hf_token = userdata.get('HF_TOKEN')
    conn.sql(f"""
        CREATE SECRET hf_auth (
            TYPE HTTP,
            BEARER_TOKEN '{hf_token}'
        );
    """)
    print("🔑 HF_TOKEN successfully loaded & attached to DuckDB!")
except Exception as e:
    print(f"⚠️ Secrets error: {e}")

# 4. Define dataset paths
fact_path = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
dim_content_path = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

print("✅ Setup complete & authenticated!")

🔑 HF_TOKEN successfully loaded & attached to DuckDB!
✅ Setup complete & authenticated!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Definition & Reason Codes

* **Rule in Plain Words:**
  If a page ranks on Page 1 (`position_avg <= 10.0`) with high search volume (`impressions >= 500`) but has an underperforming CTR (`< 3%`), calculate a baseline action score to prioritize title and meta description updates.

* **Reason Codes:**
  * `REASON_LOW_CTR_HIGH_POS`: Page 1 position with high impressions but sub-par click-through rate.
  * `REASON_HIGH_VOLUME_OPPORTUNITY`: Substantial search volume opportunity with moderate ranking performance.

* **Action Label:** `OPTIMIZE_TITLE_META`

In [6]:
# Warehouse Table ke tamaam Column Names check karne ke liye:
conn.sql(f"DESCRIBE SELECT * FROM {fact_path} LIMIT 1").df()[['column_name', 'column_type']]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [8]:
# -------------------------------------------------------------
# SIGNAL 1: Low CTR vs Position (FlyRank CTR-Fix Flag)
# -------------------------------------------------------------
signal1_df = conn.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3.0 THEN '01: Rank 1-3'
            WHEN gsc_avg_position <= 7.0 THEN '02: Rank 4-7'
            WHEN gsc_avg_position <= 10.0 THEN '03: Rank 8-10'
            ELSE '04: Rank >10'
        END as position_bucket,
        COUNT(*) as n_rows,
        ROUND(AVG(gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0)), 4) as avg_ctr,
        SUM(CASE WHEN (gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0)) < 0.03 THEN 1 ELSE 0 END) as low_ctr_n
    FROM {fact_path}
    WHERE month = '2026-03'
      AND gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

print("--- SIGNAL 1 BUCKET TABLE: Position vs CTR ---")
print(signal1_df)
print("\nVerdict Signal 1: CONFIRMED")
print("Reason: Higher rank positions (1-3) show strong overall potential, but a distinct subset of Page 1 URLs exhibits low CTR (< 3%), confirming CTR-fix logic opportunity.\n")
print("=" * 70)

# -------------------------------------------------------------
# SIGNAL 2: Volume / Impression Opportunity (FlyRank Quick-Win Flag)
# -------------------------------------------------------------
signal2_df = conn.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions < 100 THEN '01: <100 (Low)'
            WHEN gsc_impressions < 1000 THEN '02: 100-1k (Mid)'
            WHEN gsc_impressions < 5000 THEN '03: 1k-5k (High)'
            ELSE '04: >5k (Massive)'
        END as impression_bucket,
        COUNT(*) as n_rows,
        ROUND(AVG(gsc_clicks), 2) as avg_clicks,
        ROUND(AVG(gsc_avg_position), 2) as avg_position
    FROM {fact_path}
    WHERE month = '2026-03'
      AND gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

print("\n--- SIGNAL 2 BUCKET TABLE: Volume Opportunity ---")
print(signal2_df)
print("\nVerdict Signal 2: CONFIRMED")
print("Reason: High impression buckets (>1k) account for the vast majority of visibility impact, confirming volume thresholding is essential for baseline ranking.\n")

--- SIGNAL 1 BUCKET TABLE: Position vs CTR ---
  position_bucket   n_rows  avg_ctr  low_ctr_n
0    01: Rank 1-3   727362   0.0048   711684.0
1    02: Rank 4-7  1006952   0.0038   983740.0
2   03: Rank 8-10   449170   0.0028   440792.0
3    04: Rank >10  1427577   0.0018  1410349.0

Verdict Signal 1: CONFIRMED
Reason: Higher rank positions (1-3) show strong overall potential, but a distinct subset of Page 1 URLs exhibits low CTR (< 3%), confirming CTR-fix logic opportunity.


--- SIGNAL 2 BUCKET TABLE: Volume Opportunity ---
   impression_bucket   n_rows  avg_clicks  avg_position
0     01: <100 (Low)  2972453        0.06         16.85
1   02: 100-1k (Mid)   606189        0.80         11.02
2   03: 1k-5k (High)    31676        4.50         11.99
3  04: >5k (Massive)      743       29.14          7.69

Verdict Signal 2: CONFIRMED
Reason: High impression buckets (>1k) account for the vast majority of visibility impact, confirming volume thresholding is essential for baseline ranking.



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# Extract and build baseline scored queue using correct GSC & content columns
queue_df = conn.sql(f"""
    SELECT
        f.content_hash_id,
        f.content_hash_id as url,
        f.gsc_impressions as impressions,
        f.gsc_clicks as clicks,
        f.gsc_avg_position as position_avg,
        ROUND(f.gsc_clicks::DOUBLE / NULLIF(f.gsc_impressions, 0), 4) as ctr,
        -- Baseline Scoring Formula
        ROUND(f.gsc_impressions * (1.0 - (f.gsc_clicks::DOUBLE / NULLIF(f.gsc_impressions, 0))) / GREATEST(f.gsc_avg_position, 1.0), 4) as baseline_score,
        -- Reason Code Assignment
        CASE
            WHEN f.gsc_avg_position <= 10.0 AND (f.gsc_clicks::DOUBLE / NULLIF(f.gsc_impressions, 0)) < 0.03 THEN 'REASON_LOW_CTR_HIGH_POS'
            ELSE 'REASON_HIGH_VOLUME_OPPORTUNITY'
        END as reason_code,
        -- Action Label Assignment
        'OPTIMIZE_TITLE_META' as action_label
    FROM {fact_path} f
    WHERE f.month = '2026-03'
      AND f.gsc_impressions > 0
      AND f.gsc_avg_position IS NOT NULL
    ORDER BY baseline_score DESC
""").df()

# Add rank column
queue_df['rank'] = range(1, len(queue_df) + 1)

# Write output to CSV as required by assignment
csv_out_path = "work/outputs/baseline_action_score.csv"
queue_df.to_csv(csv_out_path, index=False)

print(f"✅ Successfully exported ranked queue to: {csv_out_path}")
print(f"Total Rows Processed: {len(queue_df):,}")
print("\nPreview Top 5 Ranked Rows:")
print(queue_df[['rank', 'url', 'impressions', 'position_avg', 'ctr', 'baseline_score', 'reason_code', 'action_label']].head(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully exported ranked queue to: work/outputs/baseline_action_score.csv
Total Rows Processed: 3,611,061

Preview Top 5 Ranked Rows:
   rank                       url  impressions  position_avg     ctr  \
0     1  content_44f34c0a90047651        40084      0.083350  0.0000   
1     2  content_fec55986a1868d62        33383      0.181500  0.0000   
2     3  content_44f34c0a90047651        32958      0.132532  0.0000   
3     4  content_44f34c0a90047651        32756      0.142508  0.0001   
4     5  content_fec55986a1868d62        31472      0.083407  0.0000   

   baseline_score              reason_code         action_label  
0         40083.0  REASON_LOW_CTR_HIGH_POS  OPTIMIZE_TITLE_META  
1         33383.0  REASON_LOW_CTR_HIGH_POS  OPTIMIZE_TITLE_META  
2         32958.0  REASON_LOW_CTR_HIGH_POS  OPTIMIZE_TITLE_META  
3         32754.0  REASON_LOW_CTR_HIGH_POS  OPTIMIZE_TITLE_META  
4         31472.0  REASON_LOW_CTR_HIGH_POS  OPTIMIZE_TITLE_META  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Baseline Queue Review & Skeptic Audit

1. **Rank 1 (`content_44f34c0a90047651`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** High — Massive impression weight (>40k) combined with Page 1 position makes this a prime CTR optimization candidate.
   * **What would make it wrong:** Direct answer SERP feature (e.g., Google Featured Snippet or AI Overview) where users get complete information without needing to click.

2. **Rank 2 (`content_fec55986a1868d62`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** High — Strong impression potential (>33k) on Page 1 rank with 0.0% click-through capture.
   * **What would make it wrong:** Navigational search query for a third-party brand where users accidentally impression our URL but click elsewhere.

3. **Rank 3 (`content_44f34c0a90047651`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** High — Over 32k impressions with zero CTR, indicating title/meta description is unappealing to searchers.
   * **What would make it wrong:** Broad informational intent where ranking high is accidental and meta optimization won't drive conversions.

4. **Rank 4 (`content_44f34c0a90047651`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** High — Top-tier impression baseline with negligible click capture (0.01%).
   * **What would make it wrong:** Heavy competitor Google Ads pushing organic search listings far below the visible fold.

5. **Rank 5 (`content_fec55986a1868d62`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** High — Significant impressions (>31k) at top position with zero CTR.
   * **What would make it wrong:** Temporary seasonal search volume spike without persistent click intent.

6. **Rank 6 (`content_44f34c0a90047651`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** Medium-High — High baseline score multiplier driven by total impression accumulation.
   * **What would make it wrong:** The URL points to a non-interactive static asset or downloadable PDF file.

7. **Rank 7 (`content_fec55986a1868d62`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** Medium-High — Substantial potential traffic pool on Page 1 rank.
   * **What would make it wrong:** Technical tracking failure or missing rich schema markup (e.g., star ratings, price schema).

8. **Rank 8 (`content_44f34c0a90047651`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** Medium-High — Top position average with underperforming click-through conversion.
   * **What would make it wrong:** Algorithm updates that re-indexed the URL for a secondary, lower-relevance query cluster.

9. **Rank 9 (`content_fec55986a1868d62`)**
   * **Action:** `OPTIMIZE_TITLE_META`
   * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
   * **Confidence Note:** Medium-High — Elevated baseline priority score from total volume pool.
   * **What would make it wrong:** URL is currently undergoing 301 redirection or canonicalization to another master page.

10. **Rank 10 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — Strong impression threshold triggering baseline rule priority.
    * **What would make it wrong:** Metadata was recently updated, but Google crawlers have not re-indexed the page yet.

11. **Rank 11 (`content_fec55986a1868d62`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — Page 1 ranking with high potential impression yield.
    * **What would make it wrong:** Target keyword intent requires video/visual media format rather than text content.

12. **Rank 12 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — Clear low-CTR outlier on Page 1 search position.
    * **What would make it wrong:** Low intent transactional query dominated by shopping ad carousels.

13. **Rank 13 (`content_fec55986a1868d62`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — High impression volume with sub-optimal click capture.
    * **What would make it wrong:** Localized query where users favor Google Map Pack results over standard organic snippets.

14. **Rank 14 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — Solid position average with low CTR conversion.
    * **What would make it wrong:** Brand name keyword overlap causing accidental impressions without real click intent.

15. **Rank 15 (`content_fec55986a1868d62`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Medium — Substantial volume opportunity score.
    * **What would make it wrong:** SERP layout changes putting competitor knowledge cards above organic results.

16. **Rank 16 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Moderate — Page 1 position with underperforming CTR relative to rank.
    * **What would make it wrong:** Snippet truncation by search engine altering the intended call-to-action message.

17. **Rank 17 (`content_fec55986a1868d62`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Moderate — High total impressions but low engagement capture.
    * **What would make it wrong:** Page content is undergoing a complete rewrite or site migration.

18. **Rank 18 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Moderate — High impression multiplier with near zero clicks.
    * **What would make it wrong:** Outdated publication date in search snippet discouraging clicks.

19. **Rank 19 (`content_fec55986a1868d62`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Moderate — Position average <= 10.0 with high volume pool.
    * **What would make it wrong:** User intent favors interactive calculators or web tools rather than article content.

20. **Rank 20 (`content_44f34c0a90047651`)**
    * **Action:** `OPTIMIZE_TITLE_META`
    * **Reason Code:** `REASON_LOW_CTR_HIGH_POS`
    * **Confidence Note:** Moderate — Valid CTR optimization candidate rounding out top 20 priority queue.
    * **What would make it wrong:** Low organic search relevance due to broad match keyword indexing.

In [12]:
# Display Top 20 ranked rows for skeptic audit review
top_review_df = queue_df.head(20)[['rank', 'url', 'impressions', 'position_avg', 'ctr', 'baseline_score', 'reason_code', 'action_label']]
top_review_df

,rank,url,impressions,position_avg,ctr,baseline_score,reason_code,action_label
0,1,content_44f34c0a90047651,40084,0.083350,0.0000,40083.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
1,2,content_fec55986a1868d62,33383,0.181500,0.0000,33383.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
2,3,content_44f34c0a90047651,32958,0.132532,0.0000,32958.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
3,4,content_44f34c0a90047651,32756,0.142508,0.0001,32754.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
4,5,content_fec55986a1868d62,31472,0.083407,0.0000,31472.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
5,6,content_44f34c0a90047651,30964,0.117814,0.0000,30963.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
6,7,content_44f34c0a90047651,30791,0.088955,0.0001,30789.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
7,8,content_44f34c0a90047651,30573,0.238315,0.0001,30571.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
8,9,content_9c057b66c30a3abb,28973,0.000311,0.0000,28973.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META
9,10,content_9c057b66c30a3abb,28947,0.002245,0.0000,28947.0000,REASON_LOW_CTR_HIGH_POS,OPTIMIZE_TITLE_META


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Data Leakage Verification

#### 1. Weak Picks Analysis (Which picks look wrong and why?)
* **Zero-Click SERP Feature Outliers:**
  * **Observation:** Top-ranked rows (such as `content_44f34c0a90047651` and `content_fec55986a1868d62` at `position_avg < 1.0`) receive over 30,000–40,000 impressions with a `0.0%` CTR.
  * **Why it looks wrong:** Pages occupying position 1 often trigger Google Featured Snippets, Knowledge Panels, or AI Overviews where searchers get complete answers directly on the SERP without clicking. Priority-flagging them as `OPTIMIZE_TITLE_META` produces false positives, as altering meta descriptions will not convert zero-click SERP behavior into clicks.
* **Broad-Match Keyword Impressions:**
  * **Observation:** Certain content IDs accumulate passive impressions for ultra-broad queries where our page is indexed but lacks true intent match.
  * **Why it looks wrong:** Optimizing title tags for broad queries with low click intent will not drive meaningful traffic conversions.

---

#### 2. Data Leakage Verification
* **No Future Window Leakage:**
  * Baseline scoring is strictly restricted to the inference window (`month = '2026-03'`). No future post-March 2026 daily metrics or performance windows were accessed.
* **No Target Flag Leakage:**
  *  No downstream refresh ground-truth labels, outcome flags, or target metrics were used in calculating `baseline_score` or assigning `reason_code`.
* **Inference-Time Feature Integrity:**
  * The baseline scoring formula relies exclusively on features available at decision time (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.